In [ ]:
from datetime import datetime
from pathlib import Path
import glob
import os
import warnings

import geopandas as gpd
from IPython.display import display, HTML
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import pandas as pd

from thomas2024.calibration import assemble_thresholds, lookup_pop_at_risk, lookup_pop_exposed


display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))
pd.set_option('display.max_rows', 50)

In [ ]:
storm_set_params = {
    "STORM_constant": ("STORM", 2000, None, None),
    "STORM_CNRM-CM6-1-HR": ("STORM", 2050, "RCP8.5", "CNRM"),
    "STORM_CMCC-CM2-VHR4": ("STORM", 2050, "RCP8.5", "CMCC"),
    "STORM_EC-Earth3P-HR": ("STORM", 2050, "RCP8.5", "EC-Earth"),
    "STORM_HadGEM3-GC31-HM": ("STORM", 2050, "RCP8.5", "HadGEM3"),

    "IRIS_PRESENT": ("IRIS", 2000, None, None),
    "IRIS_SSP5-2050": ("IRIS", 2050, "SSP5-8.5", None),

    "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2010": ("CHAZ", 2010, None, "UKESM1"),
    "CHAZ_SSP-585_GCM-CESM2_epoch-2010": ("CHAZ", 2010, None, "CESM2"),
    "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2010": ("CHAZ", 2010, None, "CNRM"),
    "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2010": ("CHAZ", 2010, None, "EC-Earth3"),
    "CHAZ_SSP-585_GCM-MIROC6_epoch-2010": ("CHAZ", 2010, None, "MIROC6"),
    "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2050": ("CHAZ", 2050, "SSP5-8.5", "UKESM1"),
    "CHAZ_SSP-585_GCM-CESM2_epoch-2050": ("CHAZ", 2050, "SSP5-8.5", "CESM2"),
    "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2050": ("CHAZ", 2050, "SSP5-8.5", "CNRM"),
    "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2050": ("CHAZ", 2050, "SSP5-8.5", "EC-Earth3"),
    "CHAZ_SSP-585_GCM-MIROC6_epoch-2050": ("CHAZ", 2050, "SSP5-8.5", "MIROC6"),

    "emanuel_ssp-585_gcm-cesm2_epoch-2005": ("Emanuel", 2005, None, "CESM2"),
    "emanuel_ssp-585_gcm-cnrm6_epoch-2005": ("Emanuel", 2005, None, "CNRM6"),
    "emanuel_ssp-585_gcm-ecearth6_epoch-2005": ("Emanuel", 2005, None, "EC-Earth6"),
    "emanuel_ssp-585_gcm-miroc6_epoch-2005": ("Emanuel", 2005, None, "MIROC6"),
    "emanuel_ssp-585_gcm-ukmo6_epoch-2005": ("Emanuel", 2005, None, "UKMO6"),
    "emanuel_ssp-585_gcm-cesm2_epoch-2050": ("Emanuel", 2050, "SSP5-8.5", "CESM2"),
    "emanuel_ssp-585_gcm-cnrm6_epoch-2050": ("Emanuel", 2050, "SSP5-8.5", "CNRM6"),
    "emanuel_ssp-585_gcm-ecearth6_epoch-2050": ("Emanuel", 2050, "SSP5-8.5", "EC-Earth6"),
    "emanuel_ssp-585_gcm-miroc6_epoch-2050": ("Emanuel", 2050, "SSP5-8.5", "MIROC6"),
    "emanuel_ssp-585_gcm-ukmo6_epoch-2050": ("Emanuel", 2050, "SSP5-8.5", "UKMO6"),
}

group_colours = {
    "IRIS": "lightgreen",
    "STORM": "plum",
    "CHAZ": "pink",
    "MIT": "lightblue",
}

sample_countries = (
    "NIC",
    "GTM",
    "USA",
    "DOM",
    "CUB",
    "MEX",
    "LKA",
    "MDG",
    "MOZ",
    "MMR",
    "BGD",
    "IND",
    "TWN",
    "KOR",
    "VNM",
    "PHL",
    "JPN",
    "CHN",
)

INCOME_LEVELS = ("H", "UM", "LM", "L")

In [ ]:
# Input data paths
country_thresholds_path = Path("data/20260907_country_thresholds.csv")
income_thresholds_path = Path("data/20260907_income_thresholds.csv")
wb_income_path = Path("data/OGHIST_2026_07_15.xlsx")
results_dir = Path("/data/tc-grid/open-gira/results")

# Output paths
plot_dir = Path("plots")

thresholds_by_country = assemble_thresholds(wb_income_path, income_thresholds_path, country_thresholds_path)

metrics = pd.concat([pd.read_parquet(p) for p in sorted(glob.glob(str(results_dir / Path("power/by_country/*/network/metrics.pq"))))])
metrics = metrics.dropna(how="all")
metrics["gamma_index"] = metrics["gamma_index"].astype(float)
metrics = metrics.rename(columns={c: f"net_{c}" for c in metrics.columns})

nodes = pd.concat([pd.read_parquet(p) for p in sorted(glob.glob(str(results_dir / Path("power/by_country/*/network/nodes.geoparquet"))))])
node_pop = nodes.loc[:, ["iso_a3", "population"]].dropna().groupby("iso_a3").sum().rename(columns={"population": "total_pop"})

risk = []
for atmosphere, (model, epoch, scenario, gcm) in storm_set_params.items():
    df = pd.read_parquet(results_dir / f"power/by_storm_set/{atmosphere}/disruption/drivers.pq")
    exposure = pd.read_parquet(results_dir / f"power/by_storm_set/{atmosphere}/exposure/people_exposed_by_event.pq")
    exposure = exposure.rename(columns={c: f"exposed_pop_{c}" for c in exposure.columns if c not in ["event_id", "iso_a3"]})
    exposure = lookup_pop_exposed(exposure, thresholds_by_country)
    df = df.merge(exposure, on=["event_id", "iso_a3"], how="left")
    df["model"] = model
    df["epoch"] = epoch
    df["scenario"] = scenario
    df["gcm"] = gcm
    risk.append(lookup_pop_at_risk(df, thresholds_by_country))

df = pd.concat(risk)
df = df.merge(metrics.reset_index(), on="iso_a3")
df = df.merge(node_pop.reset_index(), on="iso_a3")
df["wind_exceedance_ms-1"] = df["max_wind_speed_ms"] - df["threshold_ms-1"]
df["pop_at_risk_ratio"] = df["pop_at_risk"] / df["total_pop"]
df["pop_fragility_ratio"] = df["pop_at_risk"] / df["exposed_pop"]
df = df[df["pop_at_risk_ratio"] > 0]
df = df[np.isfinite(df["pop_fragility_ratio"])]


In [ ]:
f, ax = plt.subplots(figsize=(5, 4))
freq, edges, patches = ax.hist(df.pop_fragility_ratio, density=True, bins=np.logspace(0, 1, 50))
ax.set_xlabel("pop_fragility_ratio")
ax.set_xscale("log")
ax.grid(which="both", alpha=0.2)
labels = ax.set_yticklabels([])


In [ ]:
MIN_POP = 1E6
MIN_OBS = 1E3
DROP_COUNTRIES = ["NIC", "GTM", "MOZ", "MDG"]
MIN_EXCEEDANCE = -24
MAX_EXCEEDANCE = 75
N_BINS = 32
QUANTILES = [0.5, 0.75, 0.9, 0.95, 0.99, 0.999]
YMAX = 20.5

obs_per_iso_a3 = df.loc[:, ["iso_a3", "pop_at_risk"]].groupby("iso_a3").count().rename(columns={"pop_at_risk": "count"})
valid_iso_a3 = pd.Series(
    list(
        set(obs_per_iso_a3[obs_per_iso_a3["count"] > MIN_OBS].index)
        & set(df[df.total_pop > MIN_POP].iso_a3.unique())
        - set(DROP_COUNTRIES)
    )
)
iso_a3_by_pop = df.loc[:, ["iso_a3", "total_pop"]].groupby("iso_a3").first()
iso_a3_by_pop = iso_a3_by_pop.loc[valid_iso_a3].sort_values("total_pop")

n_cols = 6
n_rows = len(iso_a3_by_pop) // n_cols + 1
vmin = 1
vmax = 50
f, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3 * n_rows))
cmap = plt.get_cmap("magma_r")
cmap.set_under("white")

for i, (iso_a3, ax) in enumerate(zip(iso_a3_by_pop.index, axes.ravel())):
    data = df[(df.iso_a3 == iso_a3) & (df["wind_exceedance_ms-1"] < MAX_EXCEEDANCE)]
    hb = ax.hexbin(
        data["wind_exceedance_ms-1"],
        data["pop_fragility_ratio"],
        gridsize=(N_BINS, N_BINS),
        vmin=vmin,
        vmax=vmax,
        cmap=cmap,
        extent=(MIN_EXCEEDANCE, MAX_EXCEEDANCE, 0, YMAX),
    )
    ax.set_xlim(MIN_EXCEEDANCE, MAX_EXCEEDANCE)
    ax.set_ylim(0, YMAX)

    ratio = data["pop_fragility_ratio"].dropna()
    hist_counts, hist_edges = np.histogram(
        ratio,
        bins=np.linspace(0, YMAX, 32),
    )

    x0, x1 = ax.get_xlim()
    hist_width = 0.25 * (x1 - x0)
    max_count = hist_counts.max()
    widths = hist_counts / max_count * hist_width
    ax.barh(
        (hist_edges[:-1] + hist_edges[1:]) / 2,
        widths,
        height=np.diff(hist_edges),
        left=x0,
        align="center",
        alpha=0.35,
        edgecolor="none",
        zorder=2,
    )

    if i // n_cols != n_rows - 1:
        ax.set_xticklabels([])
    if i % n_cols != 0:
        ax.set_yticklabels([])
    ax.grid(which="both", alpha=0.2)
    ax.text(0.05, 0.87, iso_a3, transform=ax.transAxes)

for ax in axes.ravel()[len(iso_a3_by_pop):]:
    ax.axis("off")

f.supxlabel("Wind exceedance [ms-1]", y=0.08)
f.supylabel("Pop. at risk / exposed pop.", x=0.08)
f.colorbar(
    hb,
    ax=axes.ravel().tolist(),
    location="right",
    pad=0.02,
    fraction=0.025,
    label="Frequency"
)
plt.subplots_adjust(bottom=0.14, left=0.12, top=0.95, right=0.85)
f.savefig(plot_dir / "pop_fragility_ratio_vs_exceedance_by_country.png", bbox_inches="tight", pad_inches=0.4)

In [ ]:
EXC = "wind_exceedance_ms-1"
RATIO = "pop_fragility_ratio"
MARGIN = 1.2
Q_TAIL = 0.9
MIN_EVENTS = 50
N_BOOT = 150
SEED = 42


def cascade_metrics(ratio, margin=MARGIN, q_tail=Q_TAIL, min_events=MIN_EVENTS):
    ratio = np.asarray(ratio, dtype=float)
    ratio = ratio[np.isfinite(ratio) & (ratio > 0)]
    if len(ratio) < min_events:
        return None
    p_cascade = float(np.mean(ratio > margin))
    mag_cascade = float(np.median(ratio[ratio > margin])) if p_cascade > 0 else 1.0
    return {
        "n_events": len(ratio),
        "p_cascade": p_cascade,
        "mag_cascade": mag_cascade,
        "q_tail": float(np.quantile(ratio, q_tail)),
        "frac_below_floor": float(np.mean(ratio < 0.95)),
    }


def fit_country(g, rng, n_boot=N_BOOT):
    r = g[RATIO].dropna().to_numpy()
    out = cascade_metrics(r)
    if out is None:
        return None

    boots = []
    for _ in range(n_boot):
        i = rng.integers(0, len(r), len(r))
        m = cascade_metrics(r[i], min_events=1)
        if m is not None:
            boots.append(m)
    b = pd.DataFrame(boots)
    for k in ("p_cascade", "mag_cascade", "q_tail"):
        out[f"{k}_lo"], out[f"{k}_hi"] = np.percentile(b[k], [5, 95])
    return out


def fit_all(df, valid_iso_a3, seed=SEED):
    rng = np.random.default_rng(seed)
    d = df[df["iso_a3"].isin(valid_iso_a3)]
    rows = {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for iso, g in d.groupby("iso_a3"):
            r = fit_country(g, rng)
            if r is None:
                continue
            r["income_level"] = g["income_level"].mode().iat[0]
            rows[iso] = r
    res = pd.DataFrame(rows).T.infer_objects()
    res.index.name = "iso_a3"
    return res


def plot_cascade_plane(res, var, ax=None):
    ax = ax or plt.subplots(figsize=(6, 5))[1]
    colours = dict(zip(INCOME_LEVELS, plt.cm.viridis(np.linspace(0.05, 0.9, len(INCOME_LEVELS)))))

    for iso, r in res.iterrows():
        x, y = r["p_cascade"], r[var]
        ax.errorbar(
            x, y,
            xerr=[[x - r["p_cascade_lo"]], [r["p_cascade_hi"] - x]],
            yerr=[[y - r[f"{var}_lo"]], [r[f"{var}_hi"] - y]],
            fmt="o", ms=6, color=colours[r["income_level"]],
            ecolor="0.7", elinewidth=0.8, capsize=0, zorder=3)
        ax.annotate(iso, (x, y), xytext=(4, 4), textcoords="offset points", fontsize=8)

    for lv in INCOME_LEVELS:
        ax.plot([], [], "o", color=colours[lv], label=lv)
    ax.legend(title="Income level", frameon=False, fontsize=8)
    _, ymax = ax.get_ylim()
    ax.set_ylim(0, ymax)
    ax.axhline(1, color="0.8", lw=0.8, zorder=1, ls="--")
    ax.set_xlabel(f"P(cascade): fraction of events with ratio > {MARGIN}")
    if var == "mag_cascade":
        ax.set_ylabel("Cascade magnitude: median ratio")
    elif var == "q_tail":
        ax.set_ylabel(f"Cascade magnitude: q{Q_TAIL * 100:.0f} ratio")
    else:
        raise ValueError(f"Do not recognise {var}")
    ax.grid(alpha=0.25)
    ax.set_frame_on(False)
    return ax


if __name__ == "__main__":
    res = fit_all(df, valid_iso_a3)
    plot_cascade_plane(res, "mag_cascade")
    plt.savefig(plot_dir / f"network_fragility_scatter_med.png", dpi=200, bbox_inches="tight")
    plot_cascade_plane(res, "q_tail")
    plt.savefig(plot_dir / f"network_fragility_scatter_q{Q_TAIL * 100:.0f}.pdf", dpi=200, bbox_inches="tight")
    plt.tight_layout()
    plt.show()

In [ ]:
res.sort_values("p_cascade", ascending=False)

In [ ]:
# Join cascade statistics with network metrics and scatter
select_net_cols = [
    "net_betweenness_mean",
    "net_global_efficiency",
    "net_algebraic_connectivity",
    "net_diameter",
    "net_avg_shortest_path_len",
    "net_modularity",
    "net_betweenness_gini",
]
net = df.groupby("iso_a3").first().loc[:, select_net_cols]
join = res.join(net.loc[:, select_net_cols])
colours = dict(zip(INCOME_LEVELS, plt.cm.viridis(np.linspace(0.05, 0.9, len(INCOME_LEVELS)))))
for var in select_net_cols:
    f, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    for iso, r in join.iterrows():
        x, y = r[var], r["p_cascade"]
        ax1.scatter(x, y, color=colours[r["income_level"]])
        ax1.annotate(iso, (x, y), xytext=(4, 4), textcoords="offset points", fontsize=8)
    for lv in INCOME_LEVELS:
        ax1.plot([], [], "o", color=colours[lv], label=lv)
    ax1.set_xlabel(var)
    ax1.set_ylabel("P(cascade)")
    ax1.grid(alpha=0.2)
    for iso, r in join.iterrows():
        x, y = r[var], r["q_tail"]
        ax2.scatter(x, y, color=colours[r["income_level"]])
        ax2.annotate(iso, (x, y), xytext=(4, 4), textcoords="offset points", fontsize=8)
    for lv in INCOME_LEVELS:
        ax2.plot([], [], "o", color=colours[lv], label=lv)
    ax2.legend(title="Income level", frameon=False, fontsize=8)
    ax2.set_xlabel(var)
    ax2.set_ylabel("Cascade magnitude: q90 ratio")
    ax2.grid(alpha=0.2)

In [ ]:
table = join.select_dtypes(include="number").drop(columns=[c for c in join.columns if c.endswith("_lo") or c.endswith("_hi")])
corr = table.corr()
mask = np.zeros_like(corr, dtype=bool)
corr[mask] = np.nan
(corr.iloc[:, ::-1]
 .style
 .background_gradient(cmap='coolwarm', axis=None, vmin=-1, vmax=1)
 .highlight_null(color='#00000000')
 .format(precision=2)
 .set_table_styles([
    {'selector': 'th.col_heading', 'props': [
        ('writing-mode', 'vertical-rl'),
        ('transform', 'rotate(180deg)'),
        ('text-align', 'left'),
        ('vertical-align', 'bottom'),
        ('white-space', 'nowrap'),
        ('height', '150px'),
    ]}
])
)